# 📅 2026-09-07 (월) 개발 노트 : 도메인·백필 완주·s8 회귀·보안 재점검 — "있다"와 "작동한다"를 구분하는 하루

## 🎯 오늘의 목표 — "운영을 남 손에 맡길 수 있는 상태로, 저장소를 남에게 보여줄 수 있는 상태로"

- [x] 자체 도메인 `hiddengemdb.com` 연결 (Cloudflare DNS → Vercel)
- [x] 백필 3/16~현재 **완주** → 게임 17,313
- [x] s8 스냅샷 — 모수 1.5배 후 회귀 확인, `요즘 뜨는` 공백 원인 발견
- [x] 운영 Redis 복구, `/ops/*` 토큰 게이트, 실제로 두드리는 헬스체크
- [x] 공개 전 보안 재점검 — 레이트 리밋 미적용·CORS·프롬프트 입출력·공개 범위
- [x] 랭킹 문구·크기, 관리자 대시보드, 회원가입 넛지
- [x] GitHub 정리(rename·비공개·핀), IMSURE·로봇팔 포트폴리오 편입


## ⭐ 21. 도메인이 "저장됐는데 없다" — 부정 응답 캐시

**증상**: Cloudflare 존 파일 Export 에 `A 76.76.21.21` 이 분명히 있는데, 권한 서버 `kevin.ns.cloudflare.com` 에 직접 물어도 NODATA. 로컬도 1.1.1.1 도 전부 "레코드 없음".

**진단 순서**: ① Vercel 이 인증서를 발급했다(= Vercel 은 봤다) ② www CNAME 은 같은 존에서 정상 응답(= 존은 맞다) ③ 이름 중복(`hiddengemdb.com.hiddengemdb.com`) 아님 ④ 존 파일에 있다. 남는 건 **캐시**.

**원인**: A 레코드 저장 *전*에 조회한 실패 응답이 리졸버에 남았고, SOA 의 최소 TTL 이 **3600초** — 부정 응답이 최대 1시간 캐시된다. 국내 ISP 는 53번 포트를 가로채기도 해서 `nslookup ... 1.1.1.1` 조차 ISP 캐시로 답이 온다.

**확정**: 클라우드 컨테이너에서 raw DNS 패킷으로 권한 서버에 직접 질의 → `76.76.21.21` 정상. 로컬 문제였다.

**교훈**: "권한 서버에 직접 물었는데 없다"가 결정적 증거가 아닐 수 있다 — 중간에서 가로채는 리졸버가 있으면. 진단은 **다른 네트워크에서** 한 번 더.

부수 발견: Git Bash 가 `/flushdns` 를 경로로 변환해 `ipconfig /flushdns` 가 실패했다. 어제 CLAUDE.md §1'' 에 박은 규칙을 내가 그대로 밟았다 — `//flushdns`.


## ⭐ 22. 헬스체크가 healthy 인데 서비스가 죽어 있었다 — 같은 날 두 번

① DB 비밀번호 교체 후 랭킹 전부 500. ② Redis 인증 실패로 캐시 전멸(`/ops/cache` → `Authentication required.` — 우리 인증이 아니라 redis-py 의 접속 실패 문구였다).

둘 다 `/health` 는 `{"status":"healthy"}`. **정적 문자열이었다.** 원인도 같았다 — Railway 참조 변수(`${{Postgres.PGPASSWORD}}`)는 값이 바뀌어도 돌고 있는 컨테이너 환경변수는 옛 값. 재배포가 값을 프로세스에 넣는 사건이다.

**조치**: `/health` 가 `SELECT 1` + `PING` 을 실제로 두드리고 `components` 로 사유를 돌려준다. HTTP 는 200 유지(Railway 헬스체크가 이 경로를 봐서 503 이면 배포가 죽는다), `status` 로 healthy/degraded 구분. 그 헬스체크가 곧바로 Redis 장애를 처음 잡아냈다.

**교훈**: 무엇을 확인하는지 모르는 헬스체크는 증거가 아니다. → C-14.


## ⭐ 23. 백필 완주 — 그리고 그 과정에서 고친 것 넷

**결과**: 13:59 시작 → 21:37 완료. 7회차, 2,796개, 7시간 37분. 오전 중단분 포함 오늘 +4,465 → **17,313**. 마지막 회차 크롤이 `2026-03-15 < 2026-03-16 → 기간 종료` 로 스스로 끝났다 — 후보 소진이 아니라 **기간 도달**. 6/5~오늘 구간은 09-04 백필이 이미 덮고 있어 119개만 추가.

**고친 것**
1. **결제 하드 한도로 죽었는데 트레이스백만** — `billing_hard_limit_reached`. 거부 사유를 한 줄 진단으로 찍고, 배치가 안 만들어졌으면 업로드된 50MB 입력 파일을 삭제(안 지우면 스토리지에 쌓인다).
2. **회차마다 같은 낭비** — 백필 구간 위쪽 최신 출시작 7,771개를 회차마다 상세 조회(1.5초/건)하고 기간 밖이라 버렸다. 검색 결과 행을 분해해 출시일을 짝지어 후보 단계에서 제외. 15회차면 15번 반복될 낭비.
3. **`up -d` 가 백필을 죽였다** — 비밀번호 절차의 컨테이너 재생성이 `exec -d` 프로세스를 함께 죽였다. 크롤 단계라 비용 0. 규칙: 환경 변경 → `up -d` → 장기 작업. 로그가 컨테이너 안에만 쌓이던 것도 volume 마운트.
4. **백필과 주간 실행이 겹쳤다** — 18:30 주간 실행(50개)이 백필과 동시에 Steam API·DB 를 두드렸다. `data/pipeline.lock` PID 락. 죽은 PID 잔해는 이어받는다.

**현황 도구**: `exec -d` 로 띄우면 자식 출력이 버려져 STEP 줄만 남았다. `run_step` 이 자식 출력을 로그로 넘기게 하고, `pipeline_status` 를 만들어 단계·경과·실행 중 프로세스·오늘 등록 건수를 한 화면에.


## ⭐ 24. s8 — 모수 1.5배 뒤 추천이 얼마나 흔들렸나 (예측 4/5 적중)

| 예측 | 실측 | |
|---|---|---|
| 취향 5개 거의 유지 | **10 시나리오 전부 10/10, 점수 변화 0.00** | 적중 |
| 신작 리그 크게 변동·카나리아 1위 50% | 5/10 유지, 1위 **서브노티카 2**(12.7만) 로 교체 | 적중, 정렬 정상 |
| steady 불변 | 10/10 | 적중 |
| 시맨틱 소폭 | 2개 7/10(교체 6건 전부 백필 신작) | 적중 |
| **rising 소폭** | **0건** | **틀림** |

**소수점까지 불변인 것이 정합성 근거다.** 신규 4,465개가 전부 `too_new` 라 발굴 모수가 그대로였으니, 바뀌었으면 그게 버그다.

**카나리아 교체는 설계대로**. 빠진 3개(21,164 / 17,165 / 15,417) < 들어온 3개(126,916 / 52,433 / 37,159). `ranking.py` docstring 에 미리 적어둔 그대로 — "더 많이 검증받은 신작이 나오면 그 게임이 새 카나리아다".

**rising 0건이 진짜 발견**. `review_history` 17,556행이 있어도 같은 게임의 **두 시점 스냅샷**이 없으면 30일 Δ 는 계산되지 않는다. 행이 아니라 시간 간격이 없는 것. 주간 실행이 쌓으면 4~5주 뒤 살아난다. → R-22. 교훈: 파생 지표를 만들 때 "몇 시점이 필요한가"를 먼저 적는다.


## ⭐ 25. 보안 재점검 — 레이트 리밋이 '정의만' 되어 있었다

저장소 공개 유지 여부를 검토하다가 훑었다. 발견 7건, 그중 둘이 심각.

**① 레이트 리밋 미적용(높음)**: `settings.RATE_LIMIT_*` 가 있고 `test_core` 가 값을 검사해 통과 중이었는데, `@limiter.limit` 이 붙은 엔드포인트는 `taste.py` 하나. LLM+임베딩을 부르는 `/games/search/semantic` 이 무제한 — 서로 다른 질의로 qa 캐시를 우회하면 호출당 비용이 그대로. 비용 가드는 상한이지 예방이 아니다.
왜 안 붙어 있었나: slowapi 데코레이터는 `request: Request` 시그니처를 요구하는데 games.py 는 `request` 를 본문 파라미터 이름으로 쓰고 있었다. 붙이면 깨진다. → Redis 고정 창 카운터를 **의존성**으로. Redis 장애 시 **통과**(fail open).
**테스트를 바꿨다**: "설정이 있는가" → "라우트에 붙었는가".

**② `/ops/*` 무인증(높음)**: `POST /ops/cache/invalidate` 한 번으로 운영 캐시 전체 삭제. `X-Ops-Token` 게이트, **미설정 시 운영 503(fail closed)**. 토큰 없이 401 실측.
같은 코드베이스에서 두 정책이 반대다 — 파괴적 작업은 막고, 비파괴적·비용상한 있는 것은 통과. "장애 시 정책"은 원칙이 아니라 그 경로에서 무엇이 더 나쁜가로 정한다.

**③ CORS(중간)**: `.*\.vercel\.app` 이 남의 Vercel 앱 전부 허용 + 세션 쿠키 SameSite=None. 프로젝트 프리뷰 패턴으로 축소.
**④ 프롬프트 입출력**: 질의를 `<query>` 로 감싸 데이터임을 명시, 제어문자·길이 제한, LLM 출력 타입·길이 검증, `reasoning` 제거.
**⑤~⑥** `/health` 예외 메시지 노출 축소, 로그의 검색어 원문 제거.
**⑦ 공개 범위**: PRD·분석 프롬프트(249줄)·few-shot 을 저장소 밖으로. 코드는 공개, 레시피는 비공개. README 에 권리 고지.

**비속어 필터는 넣지 않았다** — 검색어가 타인에게 노출되는 경로가 없다. 막을 대상이 없는 방어는 넣지 않는다.

**세 번 반복된 패턴**: 정적 헬스체크 → 무인증 `/ops` → 미적용 레이트 리밋. 셋 다 "있다"와 "작동한다"를 구분하지 않았다. → CLAUDE.md §2''' "방어는 설정이 아니라 적용을 검사한다".


## 🗂 26. 제품·정리

- **랭킹 문구**: "인지도 대비 평가", "정착 게임", "그들만의 리그" 같은 내부 용어가 화면에 그대로 나가 있었다 → 숨은 명작 / 요즘 뜨는 / 신작. 상단 메뉴·탭 크기도 상향 — 가장 먼저 보이는 게 가장 작았다.
- **남은 제품 문제**: 숨은 명작 1위가 리뷰 82건짜리 처음 듣는 게임. 리뷰 하한 30 이 너무 낮아 정렬이 "가장 무명한 것이 이긴다"가 됐다. "아직 조용한" 탭과 겹친다. 하한 30→300 실험은 예측 적고 절제로 — 내일.
- **관리자 대시보드** `/admin/dashboard/`(방문·행동·회원·데이터), **회원가입 넛지**(검색/상세 2회 뒤, 전환 이벤트 기록).
- **GitHub**: `hidden-gem`(rename)·`imsure-ai`·`so101-manipulation-ai`(rename)·`hidden-gem-devlog` 공개+핀, 습작 6개 비공개. 공개용 저장소 두 벌 README 작성.
- **포트폴리오**: IMSURE·로봇팔 편입, 핵심요약 §0 을 "네 프로젝트와 한 번의 인턴을 잇는 하나의 태도"로 재작성.


## ⭐ 27. 스팀 로그인 500 — 키는 있었는데 라이브러리가 보는 칸이 비어 있었다

도메인 전환 후 첫 스팀 로그인에서 `/accounts/steam/callback/` 이 `Server Error (500)`. 구글은 정상.

**Railway 로그가 조용했다.** 여기서 한 번 헛발질할 수 있었는데, 조용한 게 정상이었다 —
Django 기본 `LOGGING` 의 console 핸들러에는 `require_debug_true` 필터가 걸려 있어서
`DEBUG=False` 인 운영에서는 500 트레이스백을 **어디에도** 출력하지 않는다(메일 핸들러만 붙는다).
로그가 없으니 라이브러리 소스를 읽었다.

```python
# allauth/socialaccount/providers/steam/provider.py
def sociallogin_from_response(self, request, response):
    steam_id = extract_steam_id(response.identity_url)
    steam_api_key = self.app.secret          # ← client_id 가 아니라 secret
    response._extra = request_steam_account_summary(steam_api_key, steam_id)
```

`request_steam_account_summary` 는 `resp.raise_for_status()` 를 그대로 던지고, `OpenIDCallbackView.get`
은 이 호출을 감싸지 않는다. 우리 `setup_oauth.py` 는 `client_id=STEAM_API_KEY, secret=''` 로 등록했다.
**빈 키 → Steam API 403 → 콜백 500.** 키를 발급받아 넣어둔 것과, 그 키를 라이브러리가 읽는 것은 다른 일이었다.

고친 것 넷:

1. `settings.LOGGING` 명시 — `django.request` ERROR → stdout. 다음 500 은 추측이 아니라 로그로 잡는다.
2. `setup_oauth.py` 가 `secret` 에도 키를 넣는다.
3. `SafeSteamCallbackView` — `requests.RequestException` 을 잡아 `/login?error=steam_unavailable` 로.
   자격증명이 원인이었지만 **스팀이 잠시 죽어도 같은 500** 이 난다. 남의 API 를 부르는 콜백은 그 API 장애를 500 으로 흘리지 않아야 한다.
   `config/urls.py` 에서 allauth include 앞에 같은 경로로 등록 — URL 문자열이 같으니 `reverse('steam_callback')` 이 만드는 `return_to` 는 그대로다(OpenID 검증이 이걸 본다).
4. `manage.py check_oauth` — provider 별로 **그 provider 가 실제로 읽는 필드**가 비었는지, 현재 Site 가 연결됐는지, SocialApp 이 중복인지. 값은 안 찍는다(앞 4글자+길이).

로컬 검증: 라우팅이 우리 뷰로 붙었는지, `reverse` 결과가 동일 경로인지, Steam API 403 을 주입했을 때
응답이 **302 → `https://hiddengemdb.com/login?error=steam_unavailable`** 인지(콤마 없음 — 어제 사고 회귀 방지), 그 과정의 ERROR 로그가 stdout 에 찍히는지까지.

운영 DB 한 칸은 코드 배포로 안 고쳐진다 — admin 에서 Steam SocialApp 의 `Secret key` 를 채워야 끝난다.

**이번 주 세 번째 같은 모양**: 정적 헬스체크 → 미적용 레이트 리밋 → 안 읽히는 자격증명 필드.
"설정했다"와 "그 설정이 실제로 쓰인다"는 매번 다른 문제였고, 매번 확인 명령을 안 만들어 둔 게 원인이었다.
그래서 이번엔 `check_oauth` 를 같이 만들었다.


## 📋 다음 할 일

**제품**
- ⬜ 숨은 명작 리뷰 하한 30 → 100/300/500 절제 비교 (예측 먼저) → s9
- ⬜ `요즘 뜨는` — 주간 실행 4~5주 후 자동 활성. 그때 카나리아 갱신
- ⬜ 신작 리그 상위가 전부 리뷰 2만 초과 — 조용한 신작 섹션 노출 비중 재검토

**운영**
- ⬜ **admin → SocialApp → Steam → `Secret key` 에 Steam Web API Key 입력** (배포로 안 고쳐지는 DB 한 칸) → `check_oauth` 로 확인
- ⬜ Railway fastapi `OPS_TOKEN` 확인(완료), Dependabot 알림 켜져 있는지
- ⬜ django-axes(admin 무차별 대입) — 의존성+마이그레이션이라 C-12 순서로
- ⬜ 프롬프트 수정 시 `Desktop/Hidden-Gem-비공개/prompts/` 사본 갱신 (규칙 §2'''')

**구직**
- ⬜ 자소서 — 채용공고 원문·문항·마감 순으로 조립. 소재 은행은 핵심요약 §0 + 각 프로젝트 §7
- ⬜ GitHub 프로필 README, `quant-rag` README 3줄
